# 00 · Selección de especies

Arma la lista de especies a partir de lo que iNaturalist registra en el Tolima,
en vez de heredarla de `species_list.json`. Filtra por disponibilidad de fotos
para no volver a quedar con clases de 40 imágenes.

Las especies salen del Tolima; **las fotos salen del mundo entero**. Restringir
también las fotos al departamento dejaría decenas por clase, no cientos.

Escribe `species_list_tolima.json` con el `taxon_id` resuelto. Descargar por
identificador y no por nombre evita los sinónimos que ya ensuciaron el corpus
actual.

In [ ]:
import json
import time

import pandas as pd
import requests

from corpus import CACHE, LISTA_ESPECIES, MANIFIESTO

API = "https://api.inaturalist.org/v1"
LICENCIAS = "cc0,cc-by,cc-by-nc,cc-by-sa,cc-by-nc-sa"
LICENCIAS_OK = set(LICENCIAS.split(","))
PAUSA_API = 1.0

PLACE_TOLIMA = 12744
PLACE_MARIPOSARIO = 148769
TAXON_MARIPOSAS = 47224        # Papilionoidea, incluye Hesperiidae

MIN_FOTOS = 500                # techo alto: los filtros de calidad del 02 se comen una parte
MAX_ESPECIES = 200
MIN_OBS_TOLIMA = 2             # 1 sola observación puede ser un error de identificación
ORDEN = "obs_tolima"           # "obs_tolima" prioriza lo local, "fotos_estimadas" el volumen
MUESTRA_RATIO = 100            # observaciones por especie para medir fotos/observación

CACHE_ESPECIES = CACHE / "especies_tolima.json"

## Cliente

In [ ]:
def pedir(sesion, ruta, params, intentos=5):
    """Reintenta con espera creciente ante 429 y cortes de red."""
    for intento in range(intentos):
        try:
            respuesta = sesion.get(f"{API}/{ruta}", params=params, timeout=30)
        except requests.RequestException as e:
            print(f"    {type(e).__name__}, reintento en {2 ** intento}s")
            time.sleep(2 ** intento)
            continue
        if respuesta.status_code == 429:
            espera = float(respuesta.headers.get("Retry-After", 2 ** intento))
            print(f"    429, esperando {espera:.0f}s")
            time.sleep(espera)
            continue
        respuesta.raise_for_status()
        return respuesta.json()
    raise RuntimeError(f"{ruta} falló tras {intentos} intentos")


def cargarCache():
    return json.loads(CACHE_ESPECIES.read_text()) if CACHE_ESPECIES.exists() else {}


def guardarCache(datos):
    CACHE.mkdir(parents=True, exist_ok=True)
    CACHE_ESPECIES.write_text(json.dumps(datos, indent=1, ensure_ascii=False))


sesion = requests.Session()
sesion.headers["User-Agent"] = "butterfly-dataset/1.0"

## 1 · Qué mariposas hay en el Tolima

`species_counts` devuelve una fila por especie con sus observaciones en el
departamento y, dentro del taxón, el conteo global que sirve de primer filtro.

In [ ]:
def familiaDe(taxon):
    """La familia es el ancestro inmediatamente posterior a Papilionoidea."""
    linaje = taxon.get("ancestor_ids") or []
    posicion = linaje.index(TAXON_MARIPOSAS) + 1 if TAXON_MARIPOSAS in linaje else len(linaje)
    return linaje[posicion] if posicion < len(linaje) else 0


def especiesEn(placeId, soloConfirmadas=True):
    encontradas, pagina = {}, 1
    while True:
        filtros = {"place_id": placeId, "taxon_id": TAXON_MARIPOSAS, "rank": "species",
                   "per_page": 500, "page": pagina}
        if soloConfirmadas:
            filtros["quality_grade"] = "research"
        respuesta = pedir(sesion, "observations/species_counts", filtros)
        for fila in respuesta["results"]:
            taxon = fila["taxon"]
            encontradas[taxon["id"]] = {
                "nombre": taxon["name"],
                "comun": taxon.get("preferred_common_name", ""),
                "familia_id": familiaDe(taxon),
                "obs_tolima": fila["count"],
                "obs_global": taxon.get("observations_count", 0),
            }
        if pagina * 500 >= respuesta["total_results"]:
            return encontradas
        pagina += 1
        time.sleep(PAUSA_API)


inventario = especiesEn(PLACE_TOLIMA)
enMariposario = set(especiesEn(PLACE_MARIPOSARIO, soloConfirmadas=False))
for taxonId, datos in inventario.items():
    datos["en_mariposario"] = taxonId in enMariposario

familias = {t["id"]: t["name"] for t in pedir(
    sesion, "taxa/" + ",".join(str(f) for f in
                               {d["familia_id"] for d in inventario.values()} - {0}),
    {})["results"]}
for datos in inventario.values():
    datos["familia"] = familias.get(datos.pop("familia_id"), "?")

print(f"{len(inventario)} especies research-grade en el Tolima")
print(f"{len(enMariposario)} registradas en el predio del mariposario")
print(f"{sum(1 for d in inventario.values() if d['obs_tolima'] >= MIN_OBS_TOLIMA)} "
      f"con {MIN_OBS_TOLIMA}+ observaciones locales")

## 2 · Cuántas fotos licenciadas tiene cada una

Una llamada por especie, así que es la etapa lenta. Guarda en `cache/` después
de cada una: interrumpir y reanudar no repite trabajo.

Solo se consultan las que podrían llegar al mínimo. Con un máximo realista de
2 fotos útiles por observación, por debajo de `MIN_FOTOS / 2` no hay caso.

In [ ]:
cache = cargarCache()
candidatas = {t: d for t, d in inventario.items()
              if d["obs_tolima"] >= MIN_OBS_TOLIMA and d["obs_global"] >= MIN_FOTOS / 2}
pendientes = [t for t in candidatas if str(t) not in cache]
print(f"{len(candidatas)} candidatas · {len(pendientes)} por consultar "
      f"(~{len(pendientes) * PAUSA_API / 60:.0f} min)")

for numero, taxonId in enumerate(pendientes, start=1):
    respuesta = pedir(sesion, "observations", {
        "taxon_id": taxonId, "photo_license": LICENCIAS, "photos": "true",
        "quality_grade": "research", "per_page": 0})
    cache[str(taxonId)] = {"obs_lic": respuesta["total_results"]}
    guardarCache(cache)
    if numero % 25 == 0 or numero == len(pendientes):
        print(f"  [{numero}/{len(pendientes)}] {candidatas[taxonId]['nombre']}")
    time.sleep(PAUSA_API)

## 3 · Fotos por observación

El conteo de arriba es de observaciones, no de fotos, y cada observación trae
entre una y varias. Se mide sobre una muestra aleatoria en vez de asumir un
factor fijo, porque varía por especie.

Se cuentan solo las fotos con licencia aceptada: el filtro del servidor es por
observación, así que una válida arrastra fotos ND que no sirven.

In [ ]:
def fotosPorObservacion(taxonId):
    respuesta = pedir(sesion, "observations", {
        "taxon_id": taxonId, "photo_license": LICENCIAS, "photos": "true",
        "quality_grade": "research", "per_page": MUESTRA_RATIO, "order_by": "random"})
    observaciones = respuesta["results"]
    if not observaciones:
        return 0.0
    utiles = sum(1 for o in observaciones for f in o.get("photos", [])
                 if (f.get("license_code") or "") in LICENCIAS_OK)
    return utiles / len(observaciones)


porMedir = [t for t, d in candidatas.items()
            if cache[str(t)]["obs_lic"] >= MIN_FOTOS / 2
            and "fotos_por_obs" not in cache[str(t)]]
print(f"{len(porMedir)} especies a muestrear (~{len(porMedir) * PAUSA_API / 60:.0f} min)")

for numero, taxonId in enumerate(porMedir, start=1):
    cache[str(taxonId)]["fotos_por_obs"] = fotosPorObservacion(taxonId)
    guardarCache(cache)
    if numero % 25 == 0 or numero == len(porMedir):
        print(f"  [{numero}/{len(porMedir)}]")
    time.sleep(PAUSA_API)

## 4 · Cuántas especies sobreviven al filtro

In [ ]:
tabla = pd.DataFrame([
    {"taxon_id": taxonId, **datos, **cache.get(str(taxonId), {})}
    for taxonId, datos in candidatas.items()]).fillna({"obs_lic": 0, "fotos_por_obs": 0})
tabla["fotos_estimadas"] = (tabla.obs_lic * tabla.fotos_por_obs).round().astype(int)

for minimo in (200, 300, 400, 500):
    print(f"  >= {minimo} fotos: {(tabla.fotos_estimadas >= minimo).sum()} especies")

pasan = tabla[tabla.fotos_estimadas >= MIN_FOTOS].sort_values(ORDEN, ascending=False)
print(f"\n{len(pasan)} especies superan {MIN_FOTOS} fotos; "
      f"se toman las {min(len(pasan), MAX_ESPECIES)} primeras por {ORDEN}")

seleccion = pasan.head(MAX_ESPECIES)
print(f"total estimado: {seleccion.fotos_estimadas.sum():,} fotos")
print(f"por especie: min {seleccion.fotos_estimadas.min():,}, "
      f"mediana {seleccion.fotos_estimadas.median():,.0f}, "
      f"max {seleccion.fotos_estimadas.max():,}")
print(f"en el predio del mariposario: {seleccion.en_mariposario.sum()}")
print(f"familias: {dict(seleccion.familia.value_counts())}")
seleccion[["nombre", "comun", "familia", "obs_tolima", "obs_lic", "fotos_por_obs",
           "fotos_estimadas", "en_mariposario"]].head(20)

### Qué queda fuera

Las que el Tolima registra pero iNaturalist no alcanza a surtir. Es la lista a
revisar si el mariposario tiene alguna de estas en exhibición: fotografiarlas en
el sitio es la única fuente que las recupera.

In [ ]:
# desde el inventario completo: las descartadas por el prefiltro tampoco llegan
estimadas = dict(zip(tabla.taxon_id, tabla.fotos_estimadas))
fuera = pd.DataFrame([{"taxon_id": t, **d, "fotos_estimadas": estimadas.get(t, 0)}
                      for t, d in inventario.items() if estimadas.get(t, 0) < MIN_FOTOS])
print(f"{len(fuera)} de {len(inventario)} especies del Tolima no llegan a {MIN_FOTOS} fotos")
fuera.nlargest(15, "obs_tolima")[["nombre", "familia", "obs_tolima", "obs_global",
                                  "fotos_estimadas", "en_mariposario"]]

## 5 · Qué cambia frente al corpus actual

In [ ]:
nuevas = {n.lower() for n in seleccion.nombre}

if MANIFIESTO.exists():
    actuales = {e.replace("_", " ").lower()
                for e in json.loads(MANIFIESTO.read_text())["reparto"]}
    print(f"corpus actual: {len(actuales)} especies · selección: {len(nuevas)}")
    print(f"se mantienen: {len(actuales & nuevas)} · entran: {len(nuevas - actuales)} · "
          f"salen: {len(actuales - nuevas)}")
    print("\nsalen:", ", ".join(sorted(actuales - nuevas)[:20]))
else:
    print(f"selección: {len(nuevas)} especies · no hay corpus con el que comparar")

## 6 · Escribir la lista

Lleva `taxon_id`, así que `01_scraping` puede saltarse `buscarTaxon` y descargar
por identificador. Apuntar `LISTA_ESPECIES` a este archivo es un cambio manual:
no se sobrescribe `species_list.json` para no perder la referencia del corpus
actual.

In [ ]:
salida = LISTA_ESPECIES
salida.write_text(json.dumps({
    "count": len(seleccion),
    "criterio": {"place_id": PLACE_TOLIMA, "taxon_id": TAXON_MARIPOSAS,
                 "min_fotos": MIN_FOTOS, "min_obs_tolima": MIN_OBS_TOLIMA,
                 "max_especies": MAX_ESPECIES, "orden": ORDEN},
    "species": [{"name": fila.nombre, "taxon_id": int(fila.taxon_id),
                 "familia": fila.familia,
                 "fotos_estimadas": int(fila.fotos_estimadas),
                 "en_mariposario": bool(fila.en_mariposario), "scraped": False}
                for fila in seleccion.itertuples()],
}, indent=2, ensure_ascii=False))
print(f"escrito {salida.name}: {len(seleccion)} especies")